In [ ]:
import numpy as np
import pickle as pkl
import os
from dotenv import load_dotenv
load_dotenv()
USER_PATH = '/home/XXXX/XXXX/fs_backup_feb13/'


cot = '/home/XXXX/XXXX/'

cot = np.load(open(cot, 'rb'), allow_pickle=True)



In [ ]:
USER_PATH

In [ ]:
from sklearn.utils import resample
from scipy.stats import wilcoxon
# 
outs_str = '/home/XXXX/XXXX/'

outs = np.load(open(outs_str, 'rb'), allow_pickle=True).item()
bs_outs_acc = []
outs_pred = {}
outs_acc = 0
num_trues = 0
for key, value in outs.items():
    if len(value[1]['neg']) == 0 and labels[key].strip(' ') == 'false':
        outs_pred[key] = True
        outs_acc += 1
    elif len(value[1]['pos']) == 0 and labels[key].strip(' ') == 'true':
        outs_pred[key] = True
        outs_acc += 1
    else:
        outs_pred[key] = False
    if labels[key] == 'true':
        num_trues += 1
outs_acc /= len(outs_pred.keys())
outs_pred_val = np.array(list(outs_pred.values()))

for i in range(len(outs_pred)):
    bs_outs_acc.append(np.sum(resample(outs_pred_val, n_samples=40))/40)
bs_sc = []
bs_sc_acc = []
for i in range(len(outs)):
    bs_sc.append(resample(n_votes[:len(outs)], n_samples=40))
    bs_sc_acc.append(np.sum(np.where(np.array(bs_sc[-1]) >= (np.ceil((20)/2+0.5)), 1, 0) )/len(bs_sc[-1]))
# outs['clutrr545.cnf'][1]
print(outs_acc)
print(outs_acc*len(outs_pred.keys()))
print(len(outs))

In [ ]:
print(wilcoxon(np.array(bs_outs_acc) - np.array(bs_sc_acc), alternative='greater'))

In [ ]:
from scipy import stats

confidence_level=0.95
d = bs_outs_acc
ci = stats.t.interval(confidence_level, df=len(d)-1, loc=np.mean(d), scale=np.std(d, ddof=1) / np.sqrt(len(d)))
print(ci)

In [ ]:
len(resample(outs_pred_val, n_samples=86))

In [ ]:
# USER_PATH = '/home/XXXX/XXXX/fs_backup_feb13/'
# import json
# rels = ['aunt', 'brother', 'brother-in-law', 'daughter', 'daughter-in-law','father', 'father-in-law', 'granddaughter', 'grandfather', 'grandmother', 'grandson', 'mother', 'mother-in-law', 'nephew','niece', 'sister', 'sister-in-law', 'son', 'son-in-law', 'uncle']

# from datasets import load_from_disk ; ds = load_from_disk(USER_PATH + 'LLM-project/clutrr_clean/dataset_fixed_gpt4o_graph_search/gen_train234_test2to10/test/')
# dd = ds.to_pandas().to_dict('records')
# ds = []
# import random
# for d in dd:
#     ds.append({})
#     ds[-1]['id'] =d['id']
#     ds[-1]['context'] = d['clean_story']
#     ds[-1]['query'] = d['query']
#     if len(d['graph_search_result']) == 0:
#         del ds[-1]
#         continue
#     if random.random() < 0.5:
#         ds[-1]['label'] = d['graph_search_result'][0]
#         ds[-1]['gt'] = 'true'
#     else:
#         random.shuffle(rels)
#         for rel in rels:
#             if rel not in d['graph_search_result'][0]:
#                 ds[-1]['label'] = rel
#                 ds[-1]['gt'] = 'false'
#                 break
#         # ds[-1]['label'] = 
#     # print(ds[-1])

# json.dump(ds,open(USER_PATH + 'SAT-LM/data/new_clutrr_test.json', 'w'))
    

In [ ]:
ds[0]

In [ ]:
len(cot)

In [ ]:
import shutil

def get_bb(file, del_sols=None):
    bb = {'pos':  [], 'neg': []}
    
    files = ['/'.join(file.split('/')[:-1]) + '/pos_' + file.split('/')[-1], '/'.join(file.split('/')[:-1]) + '/neg_' + file.split('/')[-1] ]
    for i in range(len(files)):
        file = files[i]
        shutil.copy(file, '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
        if not del_sols==None:
            if 'pos' in file:
                if 'neg' in file:
                    print('l. 416 uh oh')
                      
                ds = del_sols['pos']
            elif 'neg' in file:
                ds = del_sols['neg']
            for sol in ds:
                add_clause('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
                cf = open(f'/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]), 'a')
                write_str = '\n'
                for lit in sol:
                    write_str += str(-lit) + ' '
                # write_str += '0'
                cf.write(write_str)
                cf.close()
        # print('running cadical')
        os.system("timeout 5000 " + USER_PATH + "/LLM-project/cadiback/cadiback " + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]) + '> '  + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone")
        #   
        bbone= open('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone", 'r')
        lines = bbone.readlines()
        #   
        for line in lines:
            if line.startswith('b'):
                #   
                lits = line.split(' ')[1:]
                for lit in lits:
                    lit = lit.strip()
                    if lit == '0':
                        continue
                    lit = int(lit)
                    if 'pos' in file:                                
                        if 'neg' in file:
                            print('l. 447 uh oh')
                              
                        bb['pos'].append(lit)
                    elif 'neg' in file:
                            bb['neg'].append(lit)

    return bb
c = USER_PATH + '/LLM-project/dimacs_pronto_csvs/solver_finished.csv'
import csv
import json
dataset = USER_PATH + '/SAT-LM/data/pronto_test.json'

with open(dataset, 'r') as df:
    data = json.loads(df.read())
# breakpoint()
task = 'pronto'
missed=False
c = open(c, 'r')
cr = csv.reader(c)
task ='pronto'
names = []
all_outs = {}
missed_list = []
labels = {}
noisy_data=[]
mistr_data=[]
bad_data = []
seedrun = 'pronto_1'
for row in cr:
    if row[2] == 'SAT' and row[3] == 'SAT':
        cnf = open(USER_PATH + '/LLM-project/dimacs_pronto/neg_'+row[1]).readlines()[0].strip('\n')
        num_clause = int(cnf.split(' ')[-1])
        if row[1] in noisy_data or row[1] in mistr_data:
            print('noisy or mistranslate')
            continue
        if task=='pronto':
            bb = get_bb(USER_PATH + '/LLM-project/dimacs_pronto/'+row[1])
            jb = set(bb['pos']).intersection(set(bb['neg']))
            if len(jb) == 0:
                print("jb = 0", USER_PATH + '/SAT-LM/tmp/' + row[1][:-4] + '.py')
                continue
        # if num_clause > 500:
            # continue
        if row[1] in bad_data:
            print('bad data')
            continue
        names.append(row[1])
        labels[row[1]] = data[int(row[1].split('proofd5')[1].split('.')[0])]['label']

In [ ]:
len(labels)

In [ ]:
len(labels)

In [ ]:
print(len(names), len(labels))

In [ ]:
cot[i]

In [ ]:
for key, value in labels.items():
    labels[key] = value.strip(' ')

In [ ]:
value

In [ ]:
int(row[1].split('clutrr')[1].split('.cnf')[0])

In [ ]:
i = 0
cot_acc = 0
cot_preds = {}
for name in names:
    if cot[i] == labels[name].strip(' '):
        cot_acc += 1
        cot_preds[key] = True
    else:
        cot_preds[key] = False
    i += 1
print(cot_acc)

In [ ]:
326/(len(names)-500)

In [ ]:
few_shot = "Facts:n[Nancy] likes to cut the hair of her daughter [Heidi].\n[Heidi]'s sister [Lorraine] went to beauty school and taught them all how to cut hair expertly. " + \
            "\nHere are some additional facts and rules we\'ve found:\nNancy is the mother of Lorraine\n If Heidi is the sister of Lorraine and Heidi is the daughter of Nancy then Nancy is the mother of Lorraine.\n" + \
            "Question: Is the following statement true: \n\"[Lorraine] is [Nancy]\'s daughter\"\nAnswer: Let\'s think step by step. \n1. We have already found that Nancy is the mother of Lorraine.\n2. If Nancy is the mother of Lorraine, then Lorraine is the daughter of Nancy.\nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts:\n[Dale] and his sister [Nancy] are decorating for a party.\n[Nancy]'s daughter [Louise] thinks the party will be fun.\n" + \
            "Here are some additional facts and rules we\'ve found:\nDale is the uncle of Louise. If Nancy is the sister of Dale and Nancy is the mother of Louise then Dale is the uncle of Louise.\n" + \
            "Question: Is the following statement true: \n\"[Louise] is not [Dales]\'s niece\"\n" + \
            "Answer: Let\'s think step by step. 1. We are given that Dale is the uncle of Louise.\n2.If Dale is the uncle of Louise, then Louise is the niece of Dale.\nTherefore, the answer is No, the statement is not true.\n***\n" + \
            "Facts: \n[Lillian] and her sister [Nancy] are the only children in their family. \n[Lillian]'s biggest accomplishment is raising her son [Douglas]. " + \
            "\nHere are some additional facts and rules we\'ve found:\nLillian is the sister of Nancy. \nIf Nancy is the sister if Lillian then Lillian is the sister of Nancy.\n" + \
            "Question: Is the following statement true: \n\"[Douglas] is [Nancy]\'s nephew\"\nAnswer: Let\'s think step by step. \n1. [Douglas] is [Lillian]\'s son. \n2. [Nancy] is [Lillian]\'s sister. " + \
            "3\n. [Douglas] is [Nancy]\'s nephew. \nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts: \n[Ashley] liked to go to the park with her granddaughter [Charlotte]. \n[Dale], [Charlotte]'s father, like to take her to the movies instead. " + \
            "\nHere are some additional facts and rules we\'ve found:\nDale is the son of Ashley. If Dale is father of Charlotte and Ashley is the grandmother of Charlotte then Dale is the son of Ashley.\n" + \
            "Question: Is the following statement true: \n\"[Ashley] is not [Dale]\'s mother\"\nAnswer: Let\'s think step by step. \n1. We are given that Dale is the son of Ashley. \n2. If Dale is the son of Ashley, then Ashley is the mother of Dale. " + \
            "\nTherefore, the answer to the question is No, the statement is ot true.\n***\n"

ans = few_shot + 'a;sldkfj;alskdjf***'
print(ans.split('***')[4])

In [ ]:
a = 'grandson_of_james\'_sibling_James_Donald_'
split = a.split('_')
rel_str = ''
for a in split[:-1]:
    rel_str += a + '-'
rel_str = rel_str[:-1]
print(rel_str)

In [ ]:
files = []
for file in os.listdir('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/'):
    if file.startswith('FewShotCOTprontoqa_8b_') and 'iter' in file:
        files.append('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/' + file)

In [ ]:
len(files)

In [ ]:
import copy
import json
dataset = '/home/XXXX/XXXX/fs_backup_feb13/SAT-LM/data/pronto_test.json'
with open(dataset, 'r') as df:
    data = json.loads(df.read())

# labels = {}

# for i in range(len(data)):
#     labels['clutrr' + str(i) + '.cnf'] = data[i]['gt']
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/FewShotCOTPRONTO_Mon_Feb_10_20.12.28_2025_iter'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/LOT_pronto_preds_8BMon_Apr_21_10.44.50_2025_iter'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/FewShotCOTprontoqa_8b_'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/LOT_pronto_preds_70BTue_Apr_22_10.55.31_2025_iter'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/mistral_FewShotCOTPRONTO_preds_Wed_Apr_30_11.26.49_2025_iter'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/mistral_FewShotCOTPRONTO_Wed_Apr_30_11.26.49_2025_iter'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/LOT_folio_pronto_preds_iter0_[5]'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/mistral_FewShotCOTPRONTO_Wed_Apr_30_11.26.49_2025_iter'
cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/mistral_FewShotCOTPRONTO_Fri_May__9_14.56.03_2025_iter'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/LOT_folio_pronto_preds_iter'



cot_pred = []
cot_pred_list = []
cot_accs = []
# data = 
for i in range(20):
    cot = np.load(open(cot_iter + str(i)  , 'rb'))
    # cot = np.load(files[i])
    # print(len(cot))
    # break
    cot_acc = 0
    cot_preds = {}
    cot_preds_list = []
    # j = 0
    for j, name in enumerate(names):
        # print(value, [i])
        # if cot[j] == labels[name].strip(' '):
        if cot[j] == data[int(name.split('proofd5')[1].split('.')[0])]['label']:
            cot_acc += 1
            cot_preds[name] = True
            cot_preds_list.append(1)
        else:
            cot_preds[name] = False
            cot_preds_list.append(0)
        # j += 1
    print(cot_acc/len(cot))
    cot_accs.append(cot_acc)
    cot_pred.append(copy.deepcopy(cot_preds))
    cot_pred_list.append(copy.deepcopy(cot_preds_list))

In [ ]:
votespath = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/mistral_FewShotCOTPRONTO_votes_Fri_May__9_14.56.03_2025_iter14'
v = np.load(open(votespath, 'rb'), allow_pickle=True)

In [ ]:
v.item()

In [ ]:
runlogpath = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/run_log_pronto_70B.txt'
runlog = open('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/run_log_pronto_70B.txt', 'r')
lines = runlog.readlines()

In [ ]:
cot_pred_list = [[]]
for line in lines:
    if 'Correct: ' in line:
        cot_pred_list[-1].append(1)
    elif 'Incorrect: (' in line:
        cot_pred_list[-1].append(0)
    elif 'Few-Shot COT Accuracy PRONTO iter' in line:
        cot_pred_list.append([])
cot_pred_list = cot_pred_list[:-1]

In [ ]:
cot_pred_list

In [ ]:
data[int(name.split('proofd5')[1].split('.')[0])]['label']

In [ ]:
cot[j]

In [ ]:
cot

In [ ]:
cot = np.load(open(cot_iter + str(i), 'rb'))
print(cot[0])

In [ ]:
print(labels[name] == cot[0])

In [ ]:
n_votes = []
sc_pred = {}
for i in range(len(cot_pred_list[0])):
# for i in range(5):
    n_votes.append(0)
    # for j in range(len(cot_pred_list)):
    for j in range(len(cot_pred_list)):
        n_votes[-1] += cot_pred_list[j][i]
sc_acc = 0
for key, value in cot_pred[0].items():
    tmp = 0
    for j in cot_pred:
        tmp+= j[key]
    if tmp >=len(cot_pred_list)/2+1: 
        sc_pred[key] = 1
        sc_acc += 1
    
    else: sc_pred[key]=0

In [ ]:
n_votes

In [ ]:
len(cot_pred_list[0])

In [ ]:
cot_pred_list[j][i]

In [ ]:
len(cot_pred_list)

In [ ]:
n_votes[0]

In [ ]:
np.ceil(1.5)

In [ ]:
len(cot_pred_list)

In [ ]:
np.unique(n_votes, return_counts=True)

In [ ]:
print(np.sum(np.where(np.array(n_votes) >= np.ceil(len(cot_pred_list)/2+0.5), 1, 0) ))
# print(np.sum(np.where(np.array(n_votes) >= 3, 1, 0) ))
# print('sc acc:',np.sum(np.where(np.array(n_votes) >= 3, 1, 0) )/len(cot))

print('sc acc:',np.sum(np.where(np.array(n_votes) >= (np.ceil(len(range(len(cot_pred_list)))/2+0.5)), 1, 0) )/len(cot))
print('cot acc:', np.mean(cot_accs)/len(cot))

In [ ]:
len(cot_pred_list)

In [ ]:
n_votes

In [ ]:
print( np.ceil(len(cot_pred_list)/2+0.5))

In [ ]:
len(cot_pred_list)


In [ ]:
print(np.sum(np.where(np.array(n_votes) >= 2, 1, 0) ))


In [ ]:
from matplotlib import pyplot as plt

plt.hist(n_votes, bins=[0,1,2,3,4,5])

In [ ]:
few_shot = "Facts:\n[Nancy] likes to cut the hair of her daughter [Heidi].\n[Heidi]'s sister [Lorraine] went to beauty school and taught them all how to cut hair expertly. " + \
            "\n\nHere are some additional facts we\'ve found:\n[Nancy] is the mother of [Lorraine]\n" + \
            "Question: Is the following statement true: \n\"[Lorraine] is [Nancy]\'s daughter\"\n" + \
            "Answer:\nLet\'s think step by step.  \n1. [Heidi] is the sister of [Lorraine]\n2. [Heidi] is the daughter of [Nancy]\n3. If [Heidi] is the sister of [Lorraine] and [Heidi] is the daughter of [Nancy] then [Nancy] is the mother of [Lorraine].\n4. If [Nancy] is the mother of [Lorraine], then [Lorraine] is the daughter of [Nancy].\nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts:\n[Dale] and his sister [Nancy] are decorating for a party.\n[Nancy]'s daughter [Louise] thinks the party will be fun.\n" + \
            "\nHere are some additional facts we\'ve found:\n[Dale] is the uncle of [Louise]\n" + \
            "Question: Is the following statement true: \n\"[Louise] is not [Dales]\'s niece\"\n" + \
            "Answer: Le\'s think step by step. \n1. [Nancy] is the sister of [Dale]. \n2. [Nancy] is the mother of [Louise]\n3. If [Nancy] is the sister of [Dale] and [Nancy] is the mother of [Louise] then [Dale] is the uncle of [Louise].\n4.If [Dale] is the uncle of [Louise], then [Louise] is the niece of [Dale].\nTherefore, the answer is No, the statement is not true.\n***\n" + \
            "Facts: \n[Lillian] and her sister [Nancy] are the only children in their family. \n[Lillian]'s biggest accomplishment is raising her son [Douglas]. " + \
            "\n\nHere are some additional facts we\'ve found:\n[Lillian] is the sister of [Nancy]\n" + \
            "Question: Is the following statement true: \n\"[Douglas] is [Nancy]\'s nephew\"\n" + \
            "Answer:\nLet\'s think step by step. \n1. [Douglas] is [Lillian]\'s son. \n2. [Nancy] is [Lillian]\'s sister. " + \
            "\n3. If [Douglas] is the son of [Lillian] and [Lillian] is the sister of [Nancy] then [Douglas] is the nephew of [Lillian]. \nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts: \n[Ashley] liked to go to the park with her granddaughter [Charlotte]. \n[Dale], [Charlotte]'s father, like to take her to the movies instead. " + \
            "\n\nHere are some additional facts we\'ve found:\n[Dale] is the son of [Ashley].\n" + \
            "Question: Is the following statement true: \n\"[Ashley] is not [Dale]\'s mother\"\n" + \
            "Answer:\nLet\'s think step by step. \n1. [Dale] is the father of [Charlotte].\n2. [Ashley] is the grandmother of [Charlotte]. \n3. If [Dale] is father of [Charlotte] and [Ashley] is the grandmother of [Charlotte] then [Dale] is the son of [Ashley].\n4. If [Dale] is the son of [Ashley], then [Ashley] is the mother of [Dale]. " + \
            "\nTherefore, the answer to the question is No, the statement is ot true.\n***\n"

In [ ]:
import pickle as pkl

In [ ]:
outs_str = '/home/XXXX/XXXX/'
outs = np.load(open(outs_str, 'rb'), allow_pickle=True).item()

In [ ]:
len(outs)

In [ ]:
# for key, value in outs.items():
#     print(value[5])

In [ ]:
for key, value in labels.items():
    labels[key] = value.strip(' ')

In [ ]:
outs_pred = {}
outs_acc = 0
num_trues = 0
true_pos = 0
false_pos = 0
true_neg = 0
false_neg = 0
n_false = 0
n_true = 0
for key, value in outs.items():
    if value[4] == True:
        if len(value[1]['neg']) == 0 and labels[key] == 'false':
            outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'true':
            outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    else:
        if len(value[1]['neg']) == 0 and labels[key] == 'true':
            outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'false':
            outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    if labels[key] == 'true':
        num_trues += 1
    
    if labels[key] == 'false':
       n_false += 1
    elif labels[key] == 'true':
        n_true += 1
outs_acc /= len(outs_pred.keys())
# outs['clutrr545.cnf'][1]
print(outs_acc)
print(outs_acc*len(outs_pred.keys()))
print(n_true, n_false)

In [ ]:
n_false = 0
n_true = 0
for key, value in labels.items():
    if labels[key] == 'false':
       n_false += 1
    elif labels[key] == 'true':
        n_true += 1

In [ ]:
name_idx = {}
i = 0
for name in names:
    name_idx[name] = i
    i += 1

In [ ]:
for key, value in labels.items():
    labels[key] = value.strip(' ')

In [ ]:
missed = pkl.load(open('//home/XXXX/XXXX/fs_backup_feb13//missed_list_' + outs_str, 'rb'))
hunh_list = pkl.load(open('/home/XXXX/XXXX/fs_backup_feb13/hunh_' + outs_str, 'rb'))

In [ ]:
temp_outs_str = '/home/XXXX/XXXX/'
temp_outs = pkl.load(open(temp_outs_str, 'rb'))
import torch
temp_outs_pred = {}
outs_acc = 0
num_trues = 0
true_pos = 0
false_pos = 0
true_neg = 0
false_neg = 0
n_false = 0
n_true = 0
for key, value in temp_outs.items():
    if value[4] == True:
        if len(value[1]['neg']) == 0 and labels[key] == 'false':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'true':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            temp_outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    else:
        if len(value[1]['neg']) == 0 and labels[key] == 'true':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'false':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            temp_outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    if labels[key] == 'true':
        num_trues += 1
    
    if labels[key] == 'false':
       n_false += 1
    elif labels[key] == 'true':
        n_true += 1
outs_acc /= len(temp_outs_pred.keys())
# outs['clutrr545.cnf'][1]
print(outs_acc)
print(outs_acc*len(temp_outs_pred.keys()))
print(n_true, n_false)

scs_70 = []
flag_70 = []
lens_70 = []
fgf_70 = []
fbf_70 = []
colors = ['r', 'g']
skips = 0
for key in list(temp_outs.keys())[:100]:
    mat = torch.stack(temp_outs[key][5]) / torch.stack(temp_outs[key][5]).sum(1).reshape(-1,1)
    mat = mat[:,0]
    if labels[key] == 'true':
        mat = torch.concatenate((torch.tensor([n_votes[name_idx[key]]/20]), mat))
    else:
        mat = torch.concatenate((torch.tensor([1-(n_votes[name_idx[key]]/20)]), mat))
    lens_70.append(len(mat)-1)
    if labels[key] == 'true':
        if mat[0] < 0.5 and mat[-1] > 0.5:
            flag_70.append(2)
            for z in range(len(mat)):
                if mat[z] > 0.5:
                    fgf_70.append(z)
                    break
        elif mat[0] > 0.5 and mat[-1] < 0.5:
            flag_70.append(3)
            for z in range(len(mat)):
                if mat[z] < 0.5:
                    fbf_70.append(z)
                    break
        else: flag_70.append(temp_outs_pred[key])
    else:
        if mat[0] > 0.5 and mat[-1] < 0.5:
            flag_70.append(2)
            for z in range(len(mat)):
                if mat[z] < 0.5:
                    fgf_70.append(z)
                    break
        elif mat[0] < 0.5 and mat[-1] > 0.5:
            flag_70.append(3)
            for z in range(len(mat)):
                if mat[z] > 0.5:
                    fbf_70.append(z)
                    break
        else: flag_70.append(temp_outs_pred[key])
    # try: flag_70.append(temp_outs_pred[key])
    # except: 
    #     skips += 1
    #     continue
    scs_70.append(mat.clone())
temp_outs_str = '/home/XXXX/XXXX/'
temp_outs = pkl.load(open(temp_outs_str, 'rb'))

temp_outs_pred = {}
outs_acc = 0
num_trues = 0
true_pos = 0
false_pos = 0
true_neg = 0
false_neg = 0
n_false = 0
n_true = 0
for key, value in temp_outs.items():
    if value[4] == True:
        if len(value[1]['neg']) == 0 and labels[key] == 'false':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'true':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            temp_outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    else:
        if len(value[1]['neg']) == 0 and labels[key] == 'true':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'false':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            temp_outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    if labels[key] == 'true':
        num_trues += 1
    
    if labels[key] == 'false':
       n_false += 1
    elif labels[key] == 'true':
        n_true += 1
outs_acc /= len(temp_outs_pred.keys())
# outs['clutrr545.cnf'][1]
print(outs_acc)
print(outs_acc*len(temp_outs_pred.keys()))
print(n_true, n_false)

scs_8 = []
flag_8 = []
lens_8 = []
first_good_flip_8= []
first_bad_flip_8 = []
colors = ['r', 'g', 'b', 'orange']
plot_labels = ['unflipped-wrong', 'unflipped-correct', 'flipped correct', 'flipped incorrect']
skips = 0
for key in list(temp_outs.keys())[:100]:
    mat = torch.stack(temp_outs[key][5]) / torch.stack(temp_outs[key][5]).sum(1).reshape(-1,1)
    mat = mat[:,0]
    if labels[key] == 'true':
        mat = torch.concatenate((torch.tensor([n_votes[name_idx[key]]/20]), mat))
    else:
        mat = torch.concatenate((torch.tensor([1-(n_votes[name_idx[key]]/20)]), mat))
    lens_8.append(len(mat)-1)
    if labels[key] == 'true':
        if mat[0] < 0.5 and mat[-1] > 0.5:
            flag_8.append(2)
            for z in range(len(mat)):
                if mat[z] > 0.5:
                    first_good_flip_8.append(z)
                    break
        elif mat[0] > 0.5 and mat[-1] < 0.5:
            flag_8.append(3)
            for z in range(len(mat)):
                if mat[z] < 0.5:
                    first_bad_flip_8.append(z)
                    break
        else: flag_8.append(temp_outs_pred[key])
    else:
        if mat[0] > 0.5 and mat[-1] < 0.5:
            flag_8.append(2)
            for z in range(len(mat)):
                if mat[z] < 0.5:
                    first_good_flip_8.append(z)
                    break
        elif mat[0] < 0.5 and mat[-1] > 0.5:
            flag_8.append(3)
            for z in range(len(mat)):
                if mat[z] > 0.5:
                    first_bad_flip_8.append(z)
                    break
        else: flag_8.append(temp_outs_pred[key])
    
    scs_8.append(mat.clone())
    
import matplotlib.patches as mpatches
from matplotlib import pyplot as plt

fig1, ax1 = plt.subplots()
for i in range(len(lens_8)):
    for j in range(lens_8[i]):
        lens_8.append(j)
for i in range(len(scs_8)):
    ax1.plot(scs_8[i], c=colors[int(flag_8[i])],label=plot_labels[int(flag_8[i])])
line = [1]
for i in range(0,6):
    line.append(1-i*0.1)
line2 = [0]
for i in range(0,6):
    line2.append(i*0.1)
ax1.plot(line,'--', c='black')
ax1.plot(line2,'--', c='black')

ax1.set_title('8B \"True\" classification confidence over iterations (n=' + str(len(scs_8))+')')
# fig5, ax5 = plt.subplots()

ax1.hist(lens_8, density=True, alpha=0.4, bins=range(8))
# flag_8.append(2)

un, flag8_counts = np.unique(flag_8, return_counts=True)
patches = []
for i in range(len(colors)):
    patches.append(mpatches.Patch(color=colors[i], label=plot_labels[i] + ' (n=' + str(flag8_counts[i]) + ')'))
ax1.legend(handles=patches)
# ax1.set_xticks(list(range(0,11)))
# ax1.set_xticklabels(list(range(1,12)))


for i in range(len(lens_70)):
    for j in range(lens_70[i]):
        lens_70.append(j)
fig2, ax2 = plt.subplots()
for i in range(len(scs_70)):
    ax2.plot(scs_70[i], c=colors[int(flag_70[i])], label=plot_labels[int(flag_70[i])])
ax2.set_title('70B \"True\" classification confidence over iterations (n=' + str(len(scs_70))+')')
# flag_70.append(3)
un, flag70_counts = np.unique(flag_70, return_counts=True)
patches = []
for i in range(len(colors)):
    patches.append(mpatches.Patch(color=colors[i], label=plot_labels[i] + ' (n=' + str(flag70_counts[i]) + ')'))
ax2.legend(handles=patches)
ax2.hist(lens_70, density=True, alpha=0.4, bins=range(8))
ax2.plot(line,'--', c='black')
ax2.plot(line2,'--', c='black')
# ax2.set_xlim(0,6) 

# ax2.set_xticks(list(range(0,11)))
# ax2.set_xticklabels(list(range(1,12)))

fig3, ax3 = plt.subplots()
ax3.hist(first_good_flip_8, label='First Good Flip Iteration')
ax3.set_title("8B - Iteration of Good Flip")
fig4, ax4 = plt.subplots()
ax4.hist(first_bad_flip_8, label='First Bad Flip Iteration')
ax4.set_title("8B - Iteration of Bad Flip")

fig5, ax5 = plt.subplots()
ax5.hist(fgf_70, label='First Good Flip Iteration')
ax5.set_title("70B - Iteration of Good Flip")
fig6, ax6 = plt.subplots()
ax6.hist(fbf_70, label='First Bad Flip Iteration')
ax6.set_title("70B - Iteration of Bad Flip")
# ax3.legend()

In [ ]:
temp_outs_str = '/home/XXXX/XXXX/'
temp_outs = pkl.load(open(temp_outs_str, 'rb'))
import torch
temp_outs_pred = {}
outs_acc = 0
num_trues = 0
true_pos = 0
false_pos = 0
true_neg = 0
false_neg = 0
n_false = 0
n_true = 0
for key, value in temp_outs.items():
    # if len(value[5]) <= 3: continue
    if value[4] == True:
        if len(value[1]['neg']) == 0 and labels[key] == 'false':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'true':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            temp_outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    else:
        if len(value[1]['neg']) == 0 and labels[key] == 'true':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'false':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            temp_outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    if labels[key] == 'true':
        num_trues += 1
    
    if labels[key] == 'false':
       n_false += 1
    elif labels[key] == 'true':
        n_true += 1
print(true_pos, true_neg, false_pos, false_neg)

In [ ]:
len(temp_outs)

In [ ]:
int(temp_outs_pred[key])

In [ ]:
value

In [ ]:
len(temp_outs)

In [ ]:
torch.stack(outs[key][5]).sum(0)

In [ ]:
# for key in outs.keys():
#     print(torch.stack(outs[key][5]))
#     print(torch.stack(outs[key][5]).sum(1))
#     print((torch.stack(outs[key][5]) / torch.stack(outs[key][5]).sum(1).reshape(-1,1)).max(-1))
#     print((torch.stack(outs[key][5]) / torch.stack(outs[key][5]).sum(1).reshape(-1,1)))

In [ ]:
hunh = []
missed_list = []
for miss in missed:
    missed_list.append(miss[0])
for hunh in hunh_list:
    missed_list.append(hunh)

In [ ]:
for key, value in outs.items():

In [ ]:
len(missed_list)

In [ ]:
miss_acc = 0
miss_sc_acc = 0
no_gains = 0
for miss in missed_list:
    if outs_pred[miss] == True:
        miss_acc += 1
    # print(outs_pred[miss])
    if sc_pred[miss] == True:
        miss_sc_acc += 1
    if sc_pred[miss] == False and sc_pred[miss] == False:
        no_gains += 1
    if sc_pred[miss] == True and outs_pred[miss] == True:
        no_gains += 1
print(miss_acc/len(missed_list))
print(miss_sc_acc/len(missed_list))
# print(no_gains/len(missed_list))
print(miss_sc_acc - miss_acc)

In [ ]:
len(missed_list)

In [ ]:
tt = []
tf = []
ft = []
ff = []
for miss in missed_list:
    if sc_pred[miss] == True and outs_pred[miss] == True:
        tt.append(miss)
    elif sc_pred[miss] == True and outs_pred[miss] == False:
        tf.append(miss)
    elif sc_pred[miss] == False and outs_pred[miss] == True:
        ft.append(miss)
    elif sc_pred[miss] == False and outs_pred[miss] == False:
        ff.append(miss)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
tt = []
tf = []
ft = []
ff = []
for name in labels.keys():
    # if name in missed_list:continue
    
    if sc_pred[name] == True and outs_pred[name] == True:
        tt.append(name)
    elif sc_pred[name] == True and outs_pred[name] == False:
        tf.append(name)
    elif sc_pred[name] == False and outs_pred[name] == True:
        ft.append(name)
    elif sc_pred[name] == False and outs_pred[name] == False:
        ff.append(name)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
tt = []
tf = []
ft = []
ff = []
for name in labels.keys():
    if name in missed_list:continue
    
    if cot_pred[0][name] == True and outs_pred[name] == True:
        tt.append(name)
    elif cot_pred[0][name] == True and outs_pred[name] == False:
        tf.append(name)
    elif cot_pred[0][name] == False and outs_pred[name] == True:
        ft.append(name)
    elif cot_pred[0][name] == False and outs_pred[name] == False:
        ff.append(name)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
tt = []
tf = []
ft = []
ff = []
for name in missed_list:
    # if name in missed_list:continue
    
    if cot_pred[0][name] == True and outs_pred[name] == True:
        tt.append(name)
    elif cot_pred[0][name] == True and outs_pred[name] == False:
        tf.append(name)
    elif cot_pred[0][name] == False and outs_pred[name] == True:
        ft.append(name)
    elif cot_pred[0][name] == False and outs_pred[name] == False:
        ff.append(name)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
print(tf)

In [ ]:
print(missed_list)

In [ ]:
scores = pkl.load(open('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/scores_temp1_thresh075_thresh05_dynFalse_fixed.pkl', 'rb'))

In [ ]:
print(tf)

In [ ]:
i = 1
for line in outs['clutrr60.cnf'][0]:
    print(line)
    if i % 4 == 3:
        # try:
            # print(line.split('known predicate: ')[1].split('. Known predicates are')[0].replace('___', line.split('\\box{ ')[1]))
        print(scores[line.split('known predicate: ')[1].split('. Known predicates are')[0].replace('___', line.split('\\box{ ')[1])])
        # except:
        #     print(line.split('known predicate: ')[1].split('. Known predicates are')[0].replace('___', line.split('\\box{ ')[1]))
            # print(line)
        #     break
    if i%4 == 0 and not str(line).startswith('calls'):
        # continue
        i = 2
        # print('hihi')

    else: i += 1
# print(outs['clutrr125.cnf'])
# print(outs.keys())

In [ ]:
print(list(scores.keys())[0])

In [ ]:
tt = []
tf = []
ft = []
ff = []
for miss in missed_list:
    if cot_pred[0][miss] == True and outs_pred[miss] == True:
        tt.append(miss)
    elif cot_pred[0][miss] == True and outs_pred[miss] == False:
        tf.append(miss)
    elif cot_pred[0][miss] == False and outs_pred[miss] == True:
        ft.append(miss)
    elif cot_pred[0][miss] == False and outs_pred[miss] == False:
        ff.append(miss)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
import torch
score_list = []
for score in scores.values():
    score_list.append(torch.stack(score))
score = torch.stack(score_list)

In [ ]:
score

In [ ]:
from matplotlib import pyplot as plt

fig1, ax1 = plt.subplots()
ax1.scatter(x=score[:,0], y=score[:,1], s=3)
ax1.set_xlabel('1 - Does the following rule seem contradictory?')
ax1.set_ylabel('Does the following rule seem contextually relevant?')

In [ ]:
outs_acc*60

In [ ]:
sc_acc

In [ ]:
for key, value in outs_pred.items():
    if key not in missed_list and value == False:
        print(key)

In [ ]:
for i in range(outs['clutrr366.cnf']):
    print(outs['clutrr366.cnf'][i])
# print(outs['clutrr366.cnf'])

In [ ]:
n = 7
for i in range(len(missed[n][1])):
    print(missed[n][1][i])
    # print('\n')

In [ ]:
outs_pred = {}
outs_acc = 0
num_trues = 0
for key, value in outs.items():
    if len(value[1]['neg']) == 0 and labels[key] == 'false':
        outs_pred[key] = True
        outs_acc += 1
    elif len(value[1]['pos']) == 0 and labels[key] == 'true':
        outs_pred[key] = True
        outs_acc += 1
    else:
        outs_pred[key] = False
    if labels[key] == 'true':
        num_trues += 1
outs_acc /= len(outs_pred.keys())
# outs['clutrr545.cnf'][1]

In [ ]:
outs_acc

In [ ]:
outs_pred

In [ ]:
labels[key]

In [ ]:
outs.keys()

In [ ]:
len()

In [ ]:
import shutil

def get_bb(file, del_sols=None):
    bb = {'pos':  [], 'neg': []}
    
    files = ['/'.join(file.split('/')[:-1]) + '/pos_' + file.split('/')[-1], '/'.join(file.split('/')[:-1]) + '/neg_' + file.split('/')[-1] ]
    for i in range(len(files)):
        file = files[i]
        shutil.copy(file, '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
        if not del_sols==None:
            if 'pos' in file:
                if 'neg' in file:
                    print('l. 416 uh oh')
                      
                ds = del_sols['pos']
            elif 'neg' in file:
                ds = del_sols['neg']
            for sol in ds:
                add_clause('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
                cf = open(f'/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]), 'a')
                write_str = '\n'
                for lit in sol:
                    write_str += str(-lit) + ' '
                # write_str += '0'
                cf.write(write_str)
                cf.close()
        # print('running cadical')
        os.system("timeout 5000 /home/XXXX/XXXX/fs_backup_feb13/LLM-project/cadiback/cadiback " + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]) + '> '  + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone")
        #   
        bbone= open('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone", 'r')
        lines = bbone.readlines()
        #   
        for line in lines:
            if line.startswith('b'):
                #   
                lits = line.split(' ')[1:]
                for lit in lits:
                    lit = lit.strip()
                    if lit == '0':
                        continue
                    lit = int(lit)
                    if 'pos' in file:                                
                        if 'neg' in file:
                            print('l. 447 uh oh')
                              
                        bb['pos'].append(lit)
                    elif 'neg' in file:
                            bb['neg'].append(lit)

    return bb


In [ ]:
import shutil

def get_bb(file, del_sols=None):
    bb = {'pos':  [], 'neg': []}
    
    files = ['/'.join(file.split('/')[:-1]) + '/pos_' + file.split('/')[-1], '/'.join(file.split('/')[:-1]) + '/neg_' + file.split('/')[-1] ]
    for i in range(len(files)):
        file = files[i]
        shutil.copy(file, '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
        if not del_sols==None:
            if 'pos' in file:
                if 'neg' in file:
                    print('l. 416 uh oh')
                      
                ds = del_sols['pos']
            elif 'neg' in file:
                ds = del_sols['neg']
            for sol in ds:
                add_clause('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
                cf = open(f'/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]), 'a')
                write_str = '\n'
                for lit in sol:
                    write_str += str(-lit) + ' '
                # write_str += '0'
                cf.write(write_str)
                cf.close()
        # print('running cadical')
        os.system("timeout 5000 /home/XXXX/XXXX/fs_backup_feb13/LLM-project/cadiback/cadiback " + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]) + '> '  + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone")
        #   
        bbone= open('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone", 'r')
        lines = bbone.readlines()
        #   
        for line in lines:
            if line.startswith('b'):
                #   
                lits = line.split(' ')[1:]
                for lit in lits:
                    lit = lit.strip()
                    if lit == '0':
                        continue
                    lit = int(lit)
                    if 'pos' in file:                                
                        if 'neg' in file:
                            print('l. 447 uh oh')
                              
                        bb['pos'].append(lit)
                    elif 'neg' in file:
                            bb['neg'].append(lit)

    return bb

c = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs_clutrr_csvs_debug/solver_finished.csv'
import csv
import json
dataset = '/home/XXXX/XXXX/fs_backup_feb13/SAT-LM/data/clutrr_test.json'
with open(dataset, 'r') as df:
    data = json.loads(df.read())

task = 'folio'
missed=False
c = open(c, 'r')
cr = csv.reader(c)
names = []
all_outs = {}
missed_list = []
labels = {}
for row in cr:
    if row[2] == 'SAT' and row[3] == 'SAT':
        cnf = open('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs_clutrr/neg_'+row[1]).readlines()[0].strip('\n')
        num_clause = int(cnf.split(' ')[-1])
       
        if task=='folio':
            bb = get_bb('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs_clutrr/'+row[1])
            jb = set(bb['pos']).intersection(set(bb['neg']))
            if len(jb) == 0:
                continue
        # if num_clause > 500:
            # continue
        names.append(int(row[1].split('clutrr')[1].split('.cnf')[0]))
        labels[row[1]] = data[int(row[1].split('clutrr')[1].split('.')[0])]['label']

In [ ]:
len(names)

In [ ]:
bad_data = []
mistr_data = []
noisy_data=[]
c = '/home/XXXX/LLM-project/dimacs_clutrr_csvs_debug/solver_finished.csv'
import csv
import json
dataset = '/home/XXXX/SAT-LM/data/clutrr_test.json'
with open(dataset, 'r') as df:
    data = json.loads(df.read())
# breakpoint()
task = 'folio'
missed=False
c = open(c, 'r')
cr = csv.reader(c)
names = []
all_outs = {}
missed_list = []
labels = {}
for row in cr:
        if row[2] == 'SAT' and row[3] == 'SAT':
            cnf = open('/home/XXXX/LLM-project/dimacs_clutrr/neg_'+row[1]).readlines()[0].strip('\n')
            num_clause = int(cnf.split(' ')[-1])
            if row[1] in noisy_data or row[1] in mistr_data:
                continue
            if task=='folio':
                bb = get_bb('/home/XXXX/LLM-project/dimacs_clutrr/'+row[1])
                jb = set(bb['pos']).intersection(set(bb['neg']))
                if len(jb) == 0:
                    continue
            # if num_clause > 500:
                # continue
            if row[1] in bad_data:
                continue
            names.append(row[1])
            labels[row[1]] = data[int(row[1].split('clutrr')[1].split('.')[0])]['label']
    #   

In [ ]:
len(names)

In [ ]:
labels

In [ ]:
labels.keys()

In [ ]:
import json
folio = json.load(open('/home/XXXX/SAT-LM/data/lutrr_test.json', 'r'))
folio[48]

In [ ]:
i = 0
cot_acc = 0
cot_preds = {}
for key, value in labels.items():
    if cot[i] == value:
        cot_acc += 1
        cot_preds[key] = True
    else:
        cot_preds[key] = False
    i += 1
print(cot_acc)

In [ ]:
flipped = 0
flipped_names = []
tf = []
ft = []
for name in names:
    if cot_preds['proofd5' + str(name) + '.cnf'] != outs_pred['proofd5' + str(name) + '.cnf']:
        flipped_names.append('proofd5' + str(name) + '.cnf')
        flipped += 1
    if cot_preds['proofd5' + str(name) + '.cnf'] == True and outs_pred['proofd5' + str(name) + '.cnf'] == False:
        tf.append('proofd5' + str(name) + '.cnf')
    if cot_preds['proofd5' + str(name) + '.cnf'] == False and outs_pred['proofd5' + str(name) + '.cnf'] == True:
        ft.append('proofd5' + str(name) + '.cnf')

print(flipped)
print(len(tf))
print(len(ft))

In [ ]:
flipped = 0
flipped_names = []
tf = []
ft = []
for name in missed_list:
    name = name[7:-4]
    if cot_preds['clutrr' + str(name) + '.cnf'] != outs_pred['clutrr' + str(name) + '.cnf']:
        flipped_names.append('clutrr' + str(name) + '.cnf')
        flipped += 1
    if cot_preds['clutrr' + str(name) + '.cnf'] == True and outs_pred['proofd5' + str(name) + '.cnf'] == False:
        tf.append('proofd5' + str(name) + '.cnf')
    if cot_preds['proofd5' + str(name) + '.cnf'] == False and outs_pred['proofd5' + str(name) + '.cnf'] == True:
        ft.append('proofd5' + str(name) + '.cnf')

print(flipped)
print(len(tf))
print(len(ft))

In [ ]:
missed

In [ ]:
print(tf)

In [ ]:
outs['proofd542.cnf']


In [ ]:
name

In [ ]:
ours = pkl.load(open('/home/XXXX/LLM-project/all_outs_temp1_dynFalse.pkl', 'rb'))

In [ ]:
print()

In [ ]:
list(ours.keys())[5]


In [ ]:
cot

In [ ]:
labels[list(ours.keys())[5]]

In [ ]:
outs_str = '/home/XXXX/XXXX'
outs_70b = pkl.load(open(outs_str, 'rb'))
sc = []
exit_n = []
for key, value in outs_70b.items():
    exit_n.append(len(value[-1]))
    # print(a[-1])
from matplotlib import pyplot as plt

plt.hist(exit_n, bins=list(range(10)))

In [ ]:
outs_str = '/home/XXXX/XXXX'
outs_8b = pkl.load(open(outs_str, 'rb'))
sc = []
exit_n = []
for key, value in outs_8b.items():
    exit_n.append(len(value[-1]))
    # print(a[-1])
from matplotlib import pyplot as plt

plt.hist(exit_n, bins=list(range(10)))

In [ ]:
outs_acc_n = {}
outs_totals_n = {}
for i in range(10):
    outs_acc_n[i] = 0
    outs_totals_n[i] = 0
outs_acc = 0
num_trues = 0
# outs_totals_n = {}
for key, value in outs_70b.items():
    outs_totals_n[len(value[-1])] += 1
    if len(value[1]['neg']) == 0 and labels[key].strip(' ') == 'false':
        # outs_pred[key] = True
        outs_acc_n[len(value[-1])] += 1
    elif len(value[1]['pos']) == 0 and labels[key].strip(' ') == 'true':
        # outs_pred[key] = True
        outs_acc_n[len(value[-1])] += 1
    # else:
        # # outs_pred[key] = False
    if labels[key] == 'true':
        num_trues += 1
# for i in range(10):
#     if outs_totals_n[i] == 0: continue
#     outs_acc_n[i] /= outs_totals_n[i]
# outs_acc_n /= len(outs_pred.keys())
# outs['clutrr545.cnf'][1]
# print(outs_acc)
# print(outs_acc*len(outs_pred.keys()))

plt.bar(x = range(10),height=list(outs_acc_n.values()))
plt.title('70B accuracy for iteration length')

In [ ]:
outs_acc_n = {}
outs_totals_n = {}
for i in range(10):
    outs_acc_n[i] = 0
    outs_totals_n[i] = 0
outs_acc = 0
num_trues = 0
# outs_totals_n = {}
for key, value in outs_8b.items():
    outs_totals_n[len(value[-1])] += 1
    if len(value[1]['neg']) == 0 and labels[key].strip(' ') == 'false':
        # outs_pred[key] = True
        outs_acc_n[len(value[-1])] += 1
    elif len(value[1]['pos']) == 0 and labels[key].strip(' ') == 'true':
        # outs_pred[key] = True
        outs_acc_n[len(value[-1])] += 1
    # else:
        # # outs_pred[key] = False
    if labels[key] == 'true':
        num_trues += 1
# for i in range(10):
#     if outs_totals_n[i] == 0: continue
#     outs_acc_n[i] /= outs_totals_n[i]
# outs_acc_n /= len(outs_pred.keys())
# outs['clutrr545.cnf'][1]
# print(outs_acc)
# print(outs_acc*len(outs_pred.keys()))

plt.bar(x = range(10),height=list(outs_acc_n.values()))
plt.title('8B accuracy for iteration length')

In [ ]:
acc = 0
for n, a in outs_acc_n.items():
    acc += outs_totals_n[n]*a
print(acc/len(outs_8b))

In [ ]:
i = 0
sc_acc_n = {}
sc70 = np.where(np.array(n_votes) >= np.ceil(len(cot_pred_list)/2+0.5), 1, 0)
for j in range(10):
    sc_acc_n[j] = 0
for key, value in outs_70b.items():
    if sc70[i] == 1:
        sc_acc_n[len(value[-1])] += 1
    i += 1
# for j in range(10):
#     if outs_totals_n[j] == 0: continue
#     sc_acc_n[j] /= outs_totals_n[j]
ax1, fig1 = plt.subplots()
fig1.bar(x=range(10), height=sc_acc_n.values(), alpha=0.3,label='SC accuracy')
fig1.bar(x=range(10), height=outs_acc_n.values(), alpha=0.3,label='Our Accuracy')


fig1.legend()
# fig1.set_title('SC-20 Llama 70B accuracy on iteration-length divisions, vs SC-5 Llama 8B our-method')
fig1.set_ylabel('Accuracy over the subset')
fig1.set_xlabel('Number of iterations of our method')
fig1.set_yticks(list(np.arange(0,1,0.1)))

In [ ]:
hybrid = outs_acc_n[1] * outs_totals_n[1] + outs_acc_n[2]* outs_totals_n[2] + outs_acc_n[3]*outs_totals_n[3] + \
    sc_acc_n[4]*outs_totals_n[4] + sc_acc_n[5]*outs_totals_n[5] + sc_acc_n[6]*outs_totals_n[6]
print(hybrid/len(outs))


In [ ]:
few_shot = "Facts:\n[Nancy] likes to cut the hair of her daughter [Heidi].\n[Heidi]'s sister [Lorraine] went to beauty school and taught them all how to cut hair expertly. " + \
            "\nHere are some additional facts and rules we\'ve found:\nNancy is the mother of Lorraine\n If [Heidi] is the sister of [Lorraine] and [Heidi] is the daughter of [Nancy] then [Nancy] is the mother of [Lorraine].\n" + \
            "Question: Is the following statement true: \n\"[Lorraine] is [Nancy]\'s daughter\"\n" + \
            "Answer:\nLet\'s think step by step.  \n1. [Heidi] is the sister of [Lorraine]\n2. [Heidi] is the daughter of [Nancy]\n3. If [Heidi] is the sister of [Lorraine] and [Heidi] is the daughter of [Nancy] then [Nancy] is the mother of [Lorraine].\n4. If [Nancy] is the mother of [Lorraine], then [Lorraine] is the daughter of [Nancy].\nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts:\n[Dale] and his sister [Nancy] are decorating for a party.\n[Nancy]'s daughter [Louise] thinks the party will be fun.\n" + \
            "Here are some additional facts and rules we\'ve found:\nDale is the uncle of Louise. If [Nancy] is the sister of [Dale] and [Nancy] is the mother of [Louise] then [Dale] is the uncle of [Louise].\n" + \
            "Question: Is the following statement true: \n\"[Louise] is not [Dales]\'s niece\"\n" + \
            "Answer: Le\'s think step by step. \n1. [Nancy] is the sister of [Dale]. \n2. [Nancy] is the mother of [Louise]\n3.  If [Nancy] is the sister of [Dale] and [Nancy] is the mother of [Louise] then [Dale] is the uncle of [Louise].\n4.If [Dale] is the uncle of [Louise], then [Louise] is the niece of [Dale].\nTherefore, the answer is No, the statement is not true.\n***\n" + \
            "Facts: \n[Lillian] and her sister [Nancy] are the only children in their family. \n[Lillian]'s biggest accomplishment is raising her son [Douglas]. " + \
            "\nHere are some additional facts and rules we\'ve found:\n[Lillian] is the sister of [Nancy]. \nIf [Nancy] is the sister if [Lillian] then [Lillian] is the sister of [Nancy].\n" + \
            "Question: Is the following statement true: \n\"[Douglas] is [Nancy]\'s nephew\"\n" + \
            "Answer:\nLet\'s think step by step. \n1. [Douglas] is [Lillian]\'s son. \n2. [Nancy] is [Lillian]\'s sister. " + \
            "\n3. If [Douglas] is the son of [Lillian] and [Lillian] is the sister of [Nancy] then [Douglas] is the nephew of [Lillian]. \nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts: \n[Ashley] liked to go to the park with her granddaughter [Charlotte]. \n[Dale], [Charlotte]'s father, like to take her to the movies instead. " + \
            "\nHere are some additional facts and rules we\'ve found:\n[Dale] is the son of [Ashley]. If [Dale] is father of [Charlotte] and [Ashley] is the grandmother of [Charlotte] then [Dale] is the son of [Ashley].\n" + \
            "Question: Is the following statement true: \n\"[Ashley] is not [Dale]\'s mother\"\n" + \
            "Answer:\nLet\'s think step by step. \n1. [Dale] is the father of [Charlotte].\n2. [Ashley] is the grandmother of [Charlotte]. \n3. If [Dale] is father of [Charlotte] and [Ashley] is the grandmother of [Charlotte] then [Dale] is the son of [Ashley].\n4. If [Dale] is the son of [Ashley], then [Ashley] is the mother of [Dale]. " + \
            "\nTherefore, the answer to the question is No, the statement is ot true.\n***\n"

print(few_shot)

In [ ]:
import torch, os
from torch.utils.data import DataLoader
os.environ["CURL_CA_BUNDLE"]=""
os.environ["REQUESTS_CA_BUNDLE"]=""
run_log_path = './run_log.txt'
run_log = open(run_log_path, 'w')
run_log.write('hello\n')
run_log.close()
unknown=False
import json
import numpy as np
import csv

In [ ]:
os.environ["CURL_CA_BUNDLE"]=""
os.environ["REQUESTS_CA_BUNDLE"]=""
os.environ["CUDA_VISIBLE_DEVICES"] = '1'
USER_PATH = '/home/XXXX/XXXX/fs_backup_feb13/'
# os.environ['TRANSFORMERS_CACHE'] = '.cache/huggingface/hub'
# cache_dir = '/ephemeral/media/data1/XXXX/hub/'
cache_dir = os.path.join(os.getcwd(), '.cache/huggingface/hub')
os.environ['TRANSFORMERS_CACHE'] = cache_dir
os.environ['HF_HOME'] = cache_dir
# import transformers

# import urllib3
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import argparse
from tqdm import tqdm
import time
import datetime

In [ ]:
import warnings
import contextlib

import requests
from urllib3.exceptions import InsecureRequestWarning

old_merge_environment_settings = requests.Session.merge_environment_settings

@contextlib.contextmanager
def no_ssl_verification():
    opened_adapters = set()

    def merge_environment_settings(self, url, proxies, stream, verify, cert):
        # Verification happens only once per connection so we need to close
        # all the opened adapters once we're done. Otherwise, the effects of
        # verify=False persist beyond the end of this context manager.
        opened_adapters.add(self.get_adapter(url))

        settings = old_merge_environment_settings(self, url, proxies, stream, verify, cert)
        settings['verify'] = False

        return settings

    requests.Session.merge_environment_settings = merge_environment_settings

    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', InsecureRequestWarning)
            yield
    finally:
        requests.Session.merge_environment_settings = old_merge_environment_settings

        for adapter in opened_adapters:
            try:
                adapter.close()
            except:
                pass

In [ ]:
class Struct:
    def __init__(self, **entries):
        self.__dict__.update(entries)

args = {'train_file_path': './example_data', 'test_file_path': './example_data', 'save_path': './../SFT_train_res', 'engine': 'meta-llama/Llama-2-13b-chat-hf', 
        'n_rows': 20, 'max_length': 1000, 'lr': 5e-05, 'weight_decay': 0.0, 'epochs': 10, 'max_grad_norm': 1.0, 'batch_size': 2, 'save_strategy': 'no', 'use_lora': True}
# args['engine'] = 'meta-llama/Meta-Llama-3-70B-Instruct'
# args['engine'] = 'meta-llama/Meta-Llama-3-70B-Instruct'
# args['engine'] = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
args['engine'] = 'Qwen/Qwen2.5-Coder-3B-Instruct'

args = Struct(**args)



In [ ]:

class LLM():
    def __init__(self, args):
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype="bfloat16",
            bnb_4bit_use_double_quant=True,
        )
        with no_ssl_verification():
            

            
            self.tokenizer = AutoTokenizer.from_pretrained(
                    args.engine
                    cache_dir = cache_dir,
                    token = os.getenv("HF_TOKEN"),
                    attn_implementation="flash_attention_2"

                    )
            self.tokenizer.pad_token = self.tokenizer.eos_token
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id            
            self.model = AutoModelForCausalLM.from_pretrained(
                    args.engine 
                    cache_dir = cache_dir,
                    quantization_config=quant_config,
                    device_map='auto',
                    token = os.getenv("HF_TOKEN"),
                    attn_implementation="flash_attention_2"
                    )

        self.tokenizer.pad_token = self.tokenizer.eos_token
    
    def sentence_probabilities(self, sentences):
        with torch.no_grad():
            sentence_tokens = self.tokenizer(sentences, return_tensors='pt', padding=True)
            sentence_token_ids = sentence_tokens.input_ids.cuda()

            # Little hack to cut down inference time by 4-5x (leads to some imprecisions when using quantization)
            # Find the common prefix and run it through the model once, to save time
            first_different_token = (sentence_token_ids == sentence_token_ids[0, :].unsqueeze(0)).all(dim=0).long().argmin()
            common_prefix = sentence_token_ids[0, :first_different_token].unsqueeze(0)
            common_prefix_output = self.model(common_prefix, use_cache=True)
            common_prefix_key_values = tuple(tuple(tensor.expand(len(sentences), -1, -1, -1) for tensor in layer) 
                                             for layer in common_prefix_output.past_key_values)

            # Process the rest of the sentences
            rest_outputs = self.model(sentence_token_ids[:, first_different_token:], past_key_values=common_prefix_key_values)
            logits = torch.concat([common_prefix_output.logits.expand(len(sentences), -1, -1), rest_outputs.logits], dim=1).cuda()
            log_probs = logits.log_softmax(-1)
            log_probs = log_probs[:, :-1, :].gather(2, sentence_token_ids[:, 1:][:, :, None]).squeeze(-1).cuda()
            log_probs = (log_probs*sentence_tokens.attention_mask.cuda()[:, 1:]).sum(-1).cpu()
        return log_probs
    def nli(self, sentences, unknown):
        # true_probs = self.sentence_probabilities(sentences + " True.")
        # false_probs = self.sentence_probabilities(sentences + " False.")
        # maybe_probs = self.sentence_probabilities(sentences + " Maybe.")
        if unknown:
            true_probs, maybe_probs, false_probs =  (self.sentence_probabilities([sentences + "(A)", sentences + "(B)", sentences + "(C)"]))
            return {'True': true_probs, 'Maybe': maybe_probs, 'False': false_probs}
        else:
            true_probs, false_probs =  (self.sentence_probabilities([sentences + "(A)", sentences + "(B)"]))
            return {'True': true_probs, 'False': false_probs}
    def yn(self, sentences, norm=True, relaxed=False, obvious=False, fewshot=None, maybe=False):
        yns = []
        for sentence in sentences:
            if fewshot:
                sentence = fewshot + sentence
            
            if relaxed:
                yns.append(sentence + "Most likely")
                yns.append(sentence + "Not necessarily")
            elif obvious:
                yns.append(sentence + "obviously true.")
                yns.append(sentence + "not obviously true.")
            elif maybe:
                yns.append(sentence + "Yes")
                yns.append(sentence + "Maybe")
                yns.append(sentence + "No")
            else:
                yns.append(sentence + "Yes")
                yns.append(sentence + "No")
        # if norm:
        #     norms = self.sentence_probabilities(sentences)
        probs = []
        batch_size = 256
        for i in range(0, len(yns), batch_size):
            if i+batch_size < len(yns):
                probs += list(self.sentence_probabilities(yns[i:i+batch_size]))
            else: 
                probs += list(self.sentence_probabilities(yns[i:]))
        probs=torch.tensor(probs)
        #   
        # probs = (self.sentence_probabilities(yns))
        # probs = torch.exp(probs)
        pyes = []
        pno = []
        pmaybe = []
        if maybe:
            z = 3
        else:
            z = 2
        for i in range(0,len(probs), z):
            # if yns[i] not in cache.keys():
                # yes, no = self.sentence_probabilities([yns[i], yns[i+1]])
            
            if maybe:
                
                yes, maybe, no = probs[i], probs[i+1], probs[i+2]
                
                      
            else:
                yes, no = probs[i], probs[i+1]
            if norm:
                if maybe: 
                    y,m,n = torch.tensor([yes, maybe, no]).softmax(-1)
                else:
                    y,n = torch.tensor([yes, no]).softmax(-1)
              
                # cache[yns[i]] = y
                # cache[yns[i+1]] = n
                pyes.append(y)
                pno.append(n)
                if maybe:
                    pmaybe.append(m)
            else:
                pyes.append(1-yes/(yes + no))
            # else:
            #     y, n = cache[yns[i]], cache[yns[i+1]]
            #     pyes.append(y)
                # pno.append(n)/
        # print('cache length', len(cache))
        # if maybe:
        
        if maybe: return torch.stack([torch.tensor(pyes), torch.tensor(pmaybe), torch.tensor(pno)])
        return torch.tensor(pyes), torch.tensor(pmaybe), torch.tensor(pno)
    def complete(self, prompt, max_new = 25, temp = 1 , topk=0):
        max_length = args.max_length
        encode_ids = self.tokenizer(
        prompt, 
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=len(prompt)+1
    ).input_ids.cuda()
        generated_outputs = self.model.generate(
        encode_ids, 
        max_new_tokens=max_new, 
        return_dict_in_generate=True, 
        output_scores=True,
        temperature=temp,
        top_k=topk
        )
        responses = self.tokenizer.batch_decode(
            generated_outputs.sequences,
            skip_special_tokens=True
        )
        return responses

In [ ]:
llm = LLM(args)

In [ ]:
import pickle as pkl
all_outs = pkl.load(open('/home/XXXX/XXXX/fs_backup_feb13/all_outs_cot_met_clutrr_rulethresh_00_cot_thresh_ANNEALING_05,_sc5,_dynamic_False,_sc5_llama_8B,_no_jb_prompt,_fixed_prmopt,_yes_rules_in_prompt,_yes_solver,_shuffled,_old_fewshut,_temp_1,_n_consec_NO-CONSEC,_sc-poo,_COPY_THAT_SAVES_ALL_COT_PROMPTS.pkl', 'rb'))

In [ ]:
n = 5
preds_70 = {}
preds_8 = {}
answers = ['true', 'false']
acc_8 = 0
acc_70 = 0
n_fewshot = 4
for key, value in all_outs.items():
    votes = torch.tensor([0,0])
    prompt = value[-1][-1]
    # breakpoint()
    for i in range(n):

        completion = llm.complete(prompt, max_new=600, temp=1)[0]
        # ans = 'Here are some facts and rules:'.join(ans.split('Here are some facts and rules:')[:5])
        # if len(ans.split('Facts')) > 7:
        #     ans = 'Facts'.join(ans.split('Facts')[:5])
        # ans_prompt = ans + "Therefore, the final answer (Yes/No) is: "
        # yn = llm.yn([ans_prompt]).values
        # nli = torch.tensor(yn[0], yn[2])
        ans = completion.split('***')[n_fewshot]

        try: lines = ans.split('\n')
        except: breakpoint()
        i = -1
        notherefore=False
        try:
            while 'Therefore' not in lines[i]:
                i -= 1
                if -1*i == len(lines):
                    notherefore=True
                    break
        except:
            breakpoint()

        # if 'Therefore' in lines[i]:
        # notherefore = True
        if not notherefore:
            if 'Yes' in lines[i]:
                nli = [1,0]
            elif 'No' in lines[i]:
                nli = [0,1]
            else:
                yn = llm.yn([ans + '\n So, is the statement true? Answer: '], maybe=True)
                nli = torch.tensor([yn[0] + yn[1]/2, yn[2] + yn[1]/2])
                print('had to yn', yn)
        else:
            yn = llm.yn([ans + '\n So, is the statement true? Answer: '], maybe=True)
            nli = torch.tensor([yn[0] + yn[1]/2, yn[2] + yn[1]/2])
            print('had to yn', yn)


        votes[torch.tensor(nli).argmax()] += 1
    print(completion)
    print('====================================')
    print(ans)
    print(votes)
    
    sc_ans = answers[votes.argmax()]
    label = labels[key].strip(' ')
    if sc_ans == label: 
        acc_70 += 1
    preds_70[key] = sc_ans
    cot_flag=True
    solout = value[1]
    if len(solout['pos'])==0 and len(solout['neg']) > 0:
            if cot_flag == True:
                preds_8[key] = 'true'
            else:
                preds_8[key] = 'false'
        
            # if preds[name] != labels[name]:
            #       
    elif len(solout['pos'])>0 and len(solout['neg']) == 0:
            if cot_flag == True:
                preds_8[key] = 'false'
    if preds_8[key] == label:
        acc_8 += 1
    print(sc_ans, preds_8[key], label, sc_ans==label)
    print('70 acc', acc_70/len(preds_70))
    print('8 acc', acc_8/len(preds_8))
    print(key)
    


In [ ]:
all_outs['clutrr355.cnf']

In [ ]:
len(preds_70)

In [ ]:
n_votes_t = torch.tensor(n_votes)/20
v_freq = {}
apv = {}
for i in range(10, 21):
    apv[i] = 0
    v_freq[i] = 0
for v in n_votes:
    if v > 10:
        apv[v] += 1
        v_freq[v] += 1
    elif v <= 10:
        v_freq[20-v] += 1

aapv = []
for key, value in apv.items():
    aapv.append(value/v_freq[key])

line = []
for i in np.arange(0, 1.05, 0.05):
    line.append(i)
plt.plot(np.arange(0.55, 1.05, 0.05), aapv[1:], label='70B SC-20 calibration')
plt.plot(np.arange(0, 1.05, 0.05),line, color='black')
plt.ylim(0,1)
plt.xlim(0, 1)
plt.legend()

In [ ]:
apv